In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile
from pathlib import Path

zip_name = next(iter(uploaded.keys()))
dataset_dir = Path("/content/bep_nha_dataset")
dataset_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(dataset_dir)

yaml_files = list(dataset_dir.rglob("data.yaml"))

for path in yaml_files:
    print(path)

In [ ]:
import yaml

if not yaml_files:
    raise FileNotFoundError(
        "Không tìm thấy data.yaml. Kiểm tra lại file ZIP."
    )

DATA_YAML = yaml_files[0]

with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

print("Dataset:", DATA_YAML)
print("Classes:", config.get("names"))
print("Train:", config.get("train"))
print("Validation:", config.get("val"))

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=8,
    patience=10,
    workers=2,
    device=0,
    project="/content/bep_nha_runs",
    name="ingredients_v1",
    seed=42
)

In [ ]:
from pathlib import Path

best_model_path = Path(
    "/content/bep_nha_runs/ingredients_v1/weights/best.pt"
)

assert best_model_path.exists(), (
    "Chưa tìm thấy best.pt. Kiểm tra log train."
)

best_model = YOLO(str(best_model_path))

metrics = best_model.val(
    data=str(DATA_YAML),
    imgsz=640,
    device=0
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

In [ ]:
from google.colab import files

uploaded_test = files.upload()
test_image = next(iter(uploaded_test.keys()))

best_model.predict(
    source=test_image,
    conf=0.25,
    save=True
)

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

root = Path("/content/bep_nha_dataset")

yaml_files = list(root.rglob("data.yaml"))
assert yaml_files, "Không tìm thấy data.yaml"

config = yaml.safe_load(yaml_files[0].read_text())
names = config["names"]

if isinstance(names, list):
    names = dict(enumerate(names))
else:
    names = {int(k): v for k, v in names.items()}

# Chỉ đếm ảnh thuộc các thư mục nhãn YOLO
label_dirs = [
    p for p in root.rglob("labels")
    if p.is_dir()
]

box_counts = Counter()
image_counts = Counter()
invalid_lines = []

for label_dir in label_dirs:
    for label_file in label_dir.rglob("*.txt"):
        seen_classes = set()

        for line_no, line in enumerate(
            label_file.read_text().splitlines(), start=1
        ):
            parts = line.strip().split()

            if not parts:
                continue

            try:
                class_id = int(parts[0])
            except ValueError:
                invalid_lines.append(
                    (str(label_file), line_no, line)
                )
                continue

            if class_id not in names:
                invalid_lines.append(
                    (str(label_file), line_no, line)
                )
                continue

            box_counts[class_id] += 1
            seen_classes.add(class_id)

        for class_id in seen_classes:
            image_counts[class_id] += 1

print("=== Thống kê class ===")
for class_id, name in names.items():
    print(
        f"{name}: "
        f"{image_counts[class_id]} ảnh, "
        f"{box_counts[class_id]} bounding boxes"
    )

print(f"\nSố dòng nhãn không hợp lệ: {len(invalid_lines)}")
for item in invalid_lines[:10]:
    print(item)

In [ ]:
from pathlib import Path

for p in Path("/content").rglob("best.pt"):
    print(p)

In [ ]:
# các class đã học được
from ultralytics import YOLO

MODEL_PATH = "/content/bep_nha_runs/ingredients_v1/weights/best.pt"

model = YOLO(MODEL_PATH)

print("Các class của model:")
print(model.names)

In [ ]:
# kiểm tra validation của từng class

from pathlib import Path

DATA_YAML = "/content/bep_nha_dataset/data.yaml"

# Tìm đúng data.yaml nếu nằm trong thư mục con
if not Path(DATA_YAML).exists():
    files = list(Path("/content/bep_nha_dataset").rglob("data.yaml"))
    print("Các file data.yaml tìm được:", files)
    DATA_YAML = str(files[0])

metrics = model.val(
    data=DATA_YAML,
    imgsz=640,
    device=0
)

print("Tên class:", metrics.names)
print("Precision từng class:", metrics.box.p)
print("Recall từng class:", metrics.box.r)
print("mAP50 từng class:", metrics.box.maps)

In [ ]:
print("Số class:", len(metrics.names))
print("Số precision:", len(metrics.box.p))
print("Số recall:", len(metrics.box.r))
print("Số mAP50:", len(metrics.box.maps))

for class_id, name in metrics.names.items():
    print(
        class_id,
        name,
        "P =", metrics.box.p[class_id]
        if class_id < len(metrics.box.p) else "N/A",
        "R =", metrics.box.r[class_id]
        if class_id < len(metrics.box.r) else "N/A",
        "mAP50 =", metrics.box.maps[class_id]
        if class_id < len(metrics.box.maps) else "N/A",
    )

In [ ]:

from ultralytics import YOLO
from IPython.display import display
from PIL import Image
from pathlib import Path

# 1. Load model đã train
model = YOLO(
    "/content/bep_nha_runs/ingredients_v1/weights/best.pt"
)

# 2. Đường dẫn ảnh cần kiểm tra
image_paths = [
    "/content/thit-bo.jpg",
    "/content/thit-lon.jpeg",
]

# 3. Kiểm tra file ảnh tồn tại
for path in image_paths:
    if not Path(path).is_file():
        raise FileNotFoundError(f"Không tìm thấy ảnh: {path}")

# 4. Chạy nhận diện
results = model.predict(
    source=image_paths,
    imgsz=640,
    conf=0.1,
    save=True,
    project="/content/bep_nha_test",
    name="meat",
    exist_ok=True,
)

# 5. Hiển thị kết quả
for result in results:
    print(f"\nẢnh: {Path(result.path).name}")

    if len(result.boxes) == 0:
        print("Không phát hiện đối tượng nào.")
    else:
        for box in result.boxes:
            class_id = int(box.cls.item())
            confidence = float(box.conf.item())

            print(
                f"{model.names[class_id]}: "
                f"{confidence:.3f}"
            )

    # Hiển thị ảnh có bounding box trực tiếp từ YOLO
    plotted_image = result.plot()
    display(Image.fromarray(plotted_image[:, :, ::-1]))

# 6. In đường dẫn thư mục đã lưu ảnh kết quả
print("\nẢnh kết quả đã lưu tại:")
print("/content/bep_nha_test/meat/")


In [ ]:
from IPython.display import Image, display

test_image = "hanh-cu.jpg"

display(Image(filename="/content/runs/detect/predict/" + test_image))

In [ ]:
from google.colab import files

files.download(str(best_model_path))